Here we have our WGAN-GP.

In [ ]:
# Imports
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import math
import matplotlib.pyplot as plt
from PIL import Image
import random
from torchvision import transforms
import torchvision.transforms.functional as TF

In [ ]:
torch.manual_seed(111)

device = ""
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
# Download and unzip dataset
!kaggle datasets download -d jahysama/animal-crossing-new-horizons-all-villagers -p data
!unzip -o data/*.zip -d data

In [ ]:
# Inspect the dataset
import os

base_path = "data/villagers" 
os.listdir(base_path)

image_paths = []

for root, _, files in os.walk(base_path):
    for f in files:
        if f.lower().endswith(".png"):
            image_paths.append(os.path.join(root, f))

sample_paths = random.sample(image_paths, 9)

plt.figure(figsize=(8, 8))

for i, path in enumerate(sample_paths):
    img = Image.open(path)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
sizes = []

for path in image_paths:
    img = Image.open(path)
    sizes.append(img.size)

widths, heights = zip(*sizes)

print("Avg width:", sum(widths)/len(widths))
print("Avg height:", sum(heights)/len(heights))

In [ ]:
# Helper functions for preprocessing data

# Add white pixels to the image to make it square.
def add_padding(img):
  w, h = img.size
  largest_side = max(w, h)

  pad_left = (largest_side - w) // 2
  pad_top = (largest_side - h) // 2

  pad_right = largest_side - w - pad_left
  pad_bottom = largest_side - h - pad_top

  padding = (pad_left, pad_top, pad_right, pad_bottom)
  return TF.pad(img, padding, fill=255)
  
# Preprocesses image and returns a tensor
def preprocess_image(img):
    img = Image.open(img)
    # Handle transparent background
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        img = bg
    else:
        img = img.convert("RGB")

    # Add padding to make image square
    square_img = add_padding(img)

    # Resize and transform image to tensor
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    return transform(img)


# Visualise the padded samples.
for i, path in enumerate(sample_paths):
    img = Image.open(path)
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        img = bg
    else:
        img = img.convert("RGB")

    plt.subplot(3, 3, i + 1)
    plt.imshow(add_padding(img))
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Data preparation
training_data = torch.stack([
    preprocess_image(p) for p in image_paths
])

print(training_data.shape)

batch_size = 62
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)


In [ ]:
# Discriminator (or critic)

# Image -> Scalar
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
        )
    def forward(self, x):
        output = self.model(x).view(-1)
        return output    
        

In [ ]:
# Generator

# Noise -> Image
class Generator(nn.Module):
    def __init__(self,latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 512 * 4 * 4),
            nn.ReLU(True),

            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1,bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1,bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.Conv2d(32, 3, 3, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        output = self.model(x)
        return output    

In [ ]:
# Hyperparameters and optimizers
latent_dim = 100
lr = 0.0002

fixed_noise = torch.randn(16, latent_dim).to(device)

discriminator = Discriminator().to(device)
generator = Generator(latent_dim=latent_dim).to(device)

optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.0, 0.9)) # cuz ADAM is bae <3
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))

In [ ]:
# Methods for saving and loading checkpoints

# Generate a directory for the checkpoints
checkpoint_dir = './checkpoints/wgan'
os.makedirs(checkpoint_dir, exist_ok=True)

# Saves the states of the Generator and Discriminator
def save_checkpoint(epoch):
    checkpoint_path = os.path.join(checkpoint_dir, f'epoch_{epoch}_checkpoint.pth')
    torch.save({
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_generator_state_dict': optimizer_generator.state_dict(),
        'optimizer_discriminator_state_dict': optimizer_discriminator.state_dict(),
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

#Loads the latest checkpoint
def load_checkpoint():
    latest_checkpoint = max([os.path.join(checkpoint_dir, f) for f in os.listdir(checkpoint_dir)], 
                            key=os.path.getctime, default=None)
    if latest_checkpoint:
        checkpoint = torch.load(latest_checkpoint)
        generator.load_state_dict(checkpoint['generator_state_dict'])
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
        optimizer_generator.load_state_dict(checkpoint['optimizer_generator_state_dict'])
        optimizer_discriminator.load_state_dict(checkpoint['optimizer_discriminator_state_dict'])
        epoch = checkpoint['epoch']
        print(f"Checkpoint restored from {latest_checkpoint}, starting at epoch {epoch + 1}")
    else:
        print("No checkpoint found. Starting training from scratch.")

In [ ]:
def DiffAugment(x):
    x = rand_brightness(x)
    x = rand_saturation(x)
    x = rand_contrast(x)
    x = rand_translation(x)
    x = rand_cutout(x)
    
    x = x.contiguous()
    return x

def rand_brightness(x):
    x = x + (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) - 0.5)
    return x

def rand_saturation(x):
    x_mean = x.mean(dim=1, keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) * 2) + x_mean
    return x

def rand_contrast(x):
    x_mean = x.mean(dim=[1, 2, 3], keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) + 0.5) + x_mean
    return x

def rand_translation(x, ratio=0.125):
    shift_x = int(x.size(2) * ratio + 0.5)
    shift_y = int(x.size(3) * ratio + 0.5)
    
    translation_x = torch.randint(-shift_x, shift_x + 1, size=[x.size(0), 1, 1], device=x.device)
    translation_y = torch.randint(-shift_y, shift_y + 1, size=[x.size(0), 1, 1], device=x.device)
    
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), device=x.device),
        torch.arange(x.size(2), device=x.device),
        torch.arange(x.size(3), device=x.device),
        indexing='ij'
    )
    
    grid_x = torch.clamp(grid_x + translation_x + shift_x, 0, x.size(2) + 2 * shift_x - 1)
    grid_y = torch.clamp(grid_y + translation_y + shift_y, 0, x.size(3) + 2 * shift_y - 1)
    
    mask = torch.ones_like(x)
    
    padding = [shift_y, shift_y, shift_x, shift_x, 0, 0, 0, 0]
    x_pad = F.pad(x, padding, value=0)
    mask_pad = F.pad(mask, padding, value=0)
    
    x_shifted = x_pad.permute(0, 2, 3, 1).contiguous()[grid_batch, grid_x, grid_y].permute(0, 3, 1, 2)
    mask_shifted = mask_pad.permute(0, 2, 3, 1).contiguous()[grid_batch, grid_x, grid_y].permute(0, 3, 1, 2)
    
    random_color = torch.rand(x.size(0), 3, 1, 1, device=x.device) * 2 - 1
    
    x = x_shifted * mask_shifted + random_color * (1 - mask_shifted)
    
    return x

def rand_cutout(x, ratio=0.5):
    cutout_size = int(x.size(2) * ratio + 0.5), int(x.size(3) * ratio + 0.5)
    offset_x = torch.randint(0, x.size(2) + (1 - cutout_size[0] % 2), size=[x.size(0), 1, 1], device=x.device)
    offset_y = torch.randint(0, x.size(3) + (1 - cutout_size[1] % 2), size=[x.size(0), 1, 1], device=x.device)
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), device=x.device),
        torch.arange(cutout_size[0], device=x.device),
        torch.arange(cutout_size[1], device=x.device),
        indexing='ij'
    )
    grid_x = torch.clamp(grid_x + offset_x - cutout_size[0] // 2, 0, x.size(2) - 1)
    grid_y = torch.clamp(grid_y + offset_y - cutout_size[1] // 2, 0, x.size(3) - 1)
    mask = torch.ones(x.size(0), x.size(2), x.size(3), dtype=x.dtype, device=x.device)
    mask[grid_batch, grid_x, grid_y] = 0

    random_color = torch.rand(x.size(0), 3, 1, 1, device=x.device) * 2 - 1

    x = x * mask.unsqueeze(1) + random_color * (1 - mask.unsqueeze(1))

    return x


In [ ]:
plt.figure(figsize=(12, 12))
    
for i, path in enumerate(sample_paths[:9]):
    img_tensor = preprocess_image(path).to(device)
    
    img_tensor = img_tensor.unsqueeze(0)
    
    aug_tensor = DiffAugment(img_tensor)
    
    img_to_show = aug_tensor.squeeze(0).cpu()
    img_to_show = img_to_show * 0.5 + 0.5 
    img_to_show = torch.clamp(img_to_show, 0, 1)
    img_to_show = img_to_show.permute(1, 2, 0).numpy()
    
    plt.subplot(3, 3, i + 1)
    plt.imshow(img_to_show)
    plt.title(f"Augmented Sample {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Training loop
g_losses = []
d_losses = []
visualization_interval = 10
n_discriminator = 5
lambda_gp = 10 # gradient penalty
num_epochs = 10

# The gradient penalty is based on the compute_gradient_penalty function found here:
#https://aneelabashir425.medium.com/medium-article-tackling-mode-collapse-in-gans-from-dcgan-to-wgan-gp-0b31c7ac3692

def gradient_penalty(discriminator, real, fake, device=device):
    batch_size = real.size(0)

    epsilon = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = epsilon * real + (1 - epsilon) * fake
    interpolated.requires_grad_(True)

    mixed_scores = discriminator(interpolated)

    gradients = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True,
    )[0]

    gradients_norm = gradients.view(batch_size, -1).norm(2, dim=1)

    gradient_penalty = torch.mean((gradients_norm - 1) ** 2)
    return gradient_penalty

for epoch in range(num_epochs*100):
    for real_images in train_dataloader:
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        
        # Train the discriminator/critic
        for _ in range(n_discriminator):
            noise = torch.randn(batch_size, latent_dim).to(device)
            synthetic_images = generator(noise)

            real_labels = DiffAugment(real_images)
            synthetic_labels = DiffAugment(synthetic_images)

            d_real = discriminator(real_labels)
            d_gen = discriminator(synthetic_labels)

            gp = gradient_penalty(discriminator,real_labels, synthetic_labels.detach())

            d_loss = -(torch.mean(d_real) - torch.mean(d_gen)) + lambda_gp * gp

            optimizer_discriminator.zero_grad()
            d_loss.backward()
            optimizer_discriminator.step()

        # Train the generator
        noise = torch.randn(batch_size, latent_dim).to(device)
        gen_images = generator(noise)

        synthetic_images_aug = DiffAugment(gen_images)

        g_loss = -torch.mean(discriminator(synthetic_images_aug))

        optimizer_generator.zero_grad()
        g_loss.backward()
        optimizer_generator.step()

    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())

    print(f"Epoch [{epoch+1}/{num_epochs}]  D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")
    
    # Visualization
    if (epoch + 1) % (visualization_interval*10) == 0:
        with torch.no_grad():
            synthetic_images = generator(fixed_noise).detach().cpu()

        grid = synthetic_images[:16]
        grid = (grid + 1) / 2  # [-1,1] → [0,1]

        fig, axes = plt.subplots(4, 4, figsize=(6, 6))
        for i, ax in enumerate(axes.flatten()):
            img = grid[i].permute(1, 2, 0)
            ax.imshow(img)
            ax.axis("off")
        plt.show()

In [ ]:
# Plot losses after training
plt.figure(figsize=(10, 5))
plt.plot(g_losses, label="Generator Loss")
plt.plot(d_losses, label="Discriminator Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Visualizes 16 random images
def show_generated_images(generator, latent_dim, num_images=16, device="cpu"):
    generator.eval()  # set to eval mode

    with torch.no_grad():
        noise = torch.randn(num_images, latent_dim).to(device)
        fake_images = generator(noise).cpu()

    # Rescale from [-1, 1] → [0, 1]
    fake_images = (fake_images + 1) / 2

    # Plot
    grid_size = int(num_images ** 0.5)
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(6, 6))

    for i, ax in enumerate(axes.flatten()):
        img = fake_images[i].permute(1, 2, 0)  # CHW → HWC
        ax.imshow(img)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_generated_images(generator,latent_dim,device=device)

In [ ]:
save_checkpoint(1000)